# Part 5: Algorithms

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 5.1 Teleportation and superdense coding

### Teleportation, with real measurements

The full protocol on three qubits: q0 holds a random message, q1 and q2 share a Bell pair. Alice's measurement is simulated with the Born rule, the state collapses, and Bob applies X if m₁ = 1 and then Z if m₀ = 1. The overlap of Bob's qubit with the message is printed for five random messages.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=9)
I, X, Z = np.eye(2), np.array([[0, 1], [1, 0]]), np.diag([1, -1])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)


def one(gate, q, n=3):
    m = np.array([[1.0]])
    for k in range(n):
        m = np.kron(m, gate if k == q else I)
    return m


def cnot(c, t, n=3):
    return one(np.diag([1, 0]), c, n) + one(np.diag([0, 1]), c, n) @ one(X, t, n)


def measure(psi, q, n=3):
    """Measure qubit q: return the outcome and the collapsed state."""
    bit = (np.arange(2 ** n) >> (n - 1 - q)) & 1          # value of qubit q in each label
    p1 = np.sum(np.abs(psi[bit == 1]) ** 2)
    m = int(rng.random() < p1)
    psi = np.where(bit == m, psi, 0)
    return m, psi / np.linalg.norm(psi)


for _ in range(5):
    v = rng.normal(size=2) + 1j * rng.normal(size=2)
    message = v / np.linalg.norm(v)
    psi = np.kron(message, [1, 0, 0, 0])                    # |message>|0>|0>
    psi = cnot(1, 2) @ one(H, 1) @ psi                      # Bell pair on q1, q2
    psi = one(H, 0) @ cnot(0, 1) @ psi                      # Alice: CNOT, then H
    m0, psi = measure(psi, 0)
    m1, psi = measure(psi, 1)
    if m1 == 1:
        psi = one(X, 2) @ psi                               # Bob's corrections
    if m0 == 1:
        psi = one(Z, 2) @ psi
    bob = psi.reshape(2, 2, 2)[m0, m1, :]                   # q2, given Alice's results
    print(f"Alice read {m0}{m1}:  overlap |<message|Bob>|^2 = {abs(np.vdot(message, bob)) ** 2:.3f}")

Alice's results change from run to run, yet Bob always ends with the message: overlap 1.000 every time.

### Exercise 5.1.1: Superdense coding

Send each of the four two-bit messages 00, 01, 10 and 11. Start from the Bell pair (|00⟩ + |11⟩)/√2. Alice applies X to q0 if the second bit is 1, then Z to q0 if the first bit is 1. Bob applies CNOT from q0 to q1, then H to q0, and measures both. Print the label Bob reads with certainty.

<details><summary>Hint</summary>

After Bob's two gates the state is a single basis state, so the label with probability 1 is `np.argmax(np.abs(psi) ** 2)`, printed in binary.

</details>

In [ ]:
# Your code here

### Exercise 5.1.2: Why CNOT cannot clone a superposition

CNOT copies basis states: |0⟩|0⟩ → |0⟩|0⟩ and |1⟩|0⟩ → |1⟩|1⟩. Apply it to |+⟩|0⟩ and compare the result with the copy |+⟩|+⟩ that a cloning machine would have to produce. How large is their overlap?

In [ ]:
# Your code here

## 5.2 Oracles: Deutsch–Jozsa and Bernstein–Vazirani

### Deutsch–Jozsa from a truth table

A function from n bits to one bit is just a list of 2<sup>n</sup> values. With the ancilla in |−⟩ the oracle multiplies each input |x⟩ by (−1)<sup>f(x)</sup>, so the whole algorithm is: start uniform, apply the signs, apply Hadamards, and read the probability of all zeros.

In [ ]:
import numpy as np

n = 3
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Hn = H
for _ in range(n - 1):
    Hn = np.kron(Hn, H)                      # H on every qubit


def prob_all_zeros(f):
    psi = Hn @ np.eye(2 ** n)[0]             # uniform superposition
    psi = (-1) ** np.array(f) * psi          # the phase oracle
    psi = Hn @ psi
    return abs(psi[0]) ** 2


functions = {
    "constant 0":           [0, 0, 0, 0, 0, 0, 0, 0],
    "constant 1":           [1, 1, 1, 1, 1, 1, 1, 1],
    "balanced, f = x0":     [0, 0, 0, 0, 1, 1, 1, 1],
    "balanced, x0x1 xor x2": [((x >> 2) & (x >> 1) & 1) ^ (x & 1) for x in range(8)],
    "neither (one 1)":      [0, 0, 0, 0, 0, 0, 0, 1],
}
for name, f in functions.items():
    print(f"{name:22s} f = {f}   P(000) = {prob_all_zeros(f):.4f}")

Constant functions give all zeros with probability 1 and balanced functions with probability 0, after a single query. The last function is neither, so the promise does not hold and the answer is in between.

### Exercise 5.2.1: Bernstein–Vazirani in one query

Hide the string s = 1011 (four bits). Start from the uniform superposition (H on all four qubits of |0000⟩), multiply the amplitude of every input x by the sign (−1)<sup>s·x</sup>, apply H on all four qubits again, and read the most likely output. It should be s itself, with probability 1.

<details><summary>Hint</summary>

s·x mod 2 is the parity of the number of positions where both s and x have a 1: `bin(s & x).count("1") % 2`.

</details>

In [ ]:
# Your code here

### Exercise 5.2.2: The Hadamard sign rule

Build H⊗H⊗H as an 8 × 8 matrix and check every entry against the rule ⟨y|H<sup>⊗n</sup>|x⟩ = (−1)<sup>x·y</sup>/√(2<sup>n</sup>), here with n = 3. Then print the signs in column x = 101: the signs of the amplitudes of H⊗H⊗H|101⟩.

<details><summary>Hint</summary>

The entry in row y and column x is `Hn[y, x]`. The parity x·y is `bin(x & y).count("1") % 2`.

</details>

In [ ]:
# Your code here

## 5.3 Grover's search

### Grover's algorithm on a list of amplitudes

Grover's algorithm needs only two operations on the vector of N amplitudes: the oracle flips the sign of the marked item, and the diffusion replaces every amplitude a by 2 × mean − a. The code runs it for N = 8 and compares the success probability after k rounds with sin²((2k + 1)θ), where sin θ = 1/√N.

In [ ]:
import numpy as np

N, marked = 8, 5
amps = np.full(N, 1 / np.sqrt(N))            # uniform superposition
theta = np.arcsin(1 / np.sqrt(N))

for k in range(1, 5):
    amps[marked] *= -1                       # oracle
    amps = 2 * amps.mean() - amps            # diffusion: inversion about the mean
    p = amps[marked] ** 2
    print(f"after {k} round{'s' if k > 1 else ' '}: P(marked) = {p:.4f}   sin^2((2k+1) theta) = {np.sin((2 * k + 1) * theta) ** 2:.4f}")

Two rounds give 94.5%, the best choice here: (2k + 1)θ is closest to 90° at k = 2, the first peak. A third round overshoots and the success probability falls to 33%.

### Exercise 5.3.1: How many rounds for 64 items?

For N = 64, compute the best number of rounds from k ≈ π/(4θ) − 1/2, then run Grover for 0 to 12 rounds and print the success probability each time. Where is the peak, and how low does it fall afterwards?

<details><summary>Hint</summary>

Reuse the two lines of the example: the oracle and the diffusion.

</details>

In [ ]:
# Your code here

### Exercise 5.3.2: Several marked items

Mark M = 4 of N = 64 items. The oracle now flips all four signs. Run 0 to 6 rounds and print the probability of measuring any marked item. Compare the best round with (π/4)√(N/M).

<details><summary>Hint</summary>

With several marked items, sin θ = √(M/N), and the success probability is the sum of the squares of the marked amplitudes.

</details>

In [ ]:
# Your code here

## 5.4 The quantum Fourier transform and phase estimation

### The QFT as a matrix

The QFT on n qubits is the N × N matrix with entries e<sup>2πi·jk/N</sup>/√N, where N = 2<sup>n</sup>. The code builds it, checks that it is unitary, and applies it to |101⟩, the number 5. NumPy's inverse FFT computes the same transform, up to a factor √N.

In [ ]:
import numpy as np

n = 3
N = 2 ** n
j, k = np.meshgrid(np.arange(N), np.arange(N))
QFT = np.exp(2j * np.pi * j * k / N) / np.sqrt(N)          # QFT[k, x] = e^{2 pi i x k / N} / sqrt(N)
print("unitary:", np.allclose(QFT.conj().T @ QFT, np.eye(N)))

x = 5
out = QFT[:, x]                                            # QFT|5> is column 5
for kk, a in enumerate(out):
    turns = (np.angle(a) / (2 * np.pi)) % 1
    print(f"|{kk:03b}>  size {abs(a):.3f}   phase {turns:.3f} of a turn   (5k/8 mod 1 = {5 * kk / 8 % 1:.3f})")
print("same as sqrt(N) * numpy.fft.ifft:", np.allclose(out, np.sqrt(N) * np.fft.ifft(np.eye(N)[x])))

Every amplitude has size 1/√8 ≈ 0.354, and the phase of |k⟩ is 5k/8 of a full turn (printed as a fraction of 2π), as in the chapter's "QFT of |101⟩" example.

### Exercise 5.4.1: QFT|x⟩ is a product state

For every x from 0 to 7, check that QFT|x⟩ equals a product of three one-qubit states (|0⟩ + e<sup>iα</sup>|1⟩)/√2, where qubit q (with q0 the leftmost) has the angle α = 2πx/2<sup>q+1</sup>.

<details><summary>Hint</summary>

Build the product with `np.kron`, one factor per qubit, and compare with column x of the QFT matrix.

</details>

In [ ]:
# Your code here

### Exercise 5.4.2: Phase estimation, outcome by outcome

After the kickback steps, the t counting qubits of phase estimation hold (1/√M) Σ<sub>k</sub> e<sup>2πiφk</sup>|k⟩, with M = 2<sup>t</sup>. Apply the inverse QFT and find the probability of every t-bit outcome, for φ = 0.375 with t = 3 and for φ = 0.3 with t = 3, 5 and 7. Print the most likely outcome as a fraction and its probability.

<details><summary>Hint</summary>

The inverse QFT is the conjugate transpose of the QFT matrix. Apply it and square the amplitudes.

</details>

In [ ]:
# Your code here

### Exercise 5.4.3: The classical half of Shor's algorithm

For N = 15 with a = 7, and for N = 21 with a = 2, find the period r of a<sup>x</sup> mod N by trying x = 1, 2, 3, …, then compute gcd(a<sup>r/2</sup> − 1, N) and gcd(a<sup>r/2</sup> + 1, N).

<details><summary>Hint</summary>

`pow(a, x, N)` computes a<sup>x</sup> mod N quickly, and `math.gcd` finds greatest common divisors.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 5.1.1: Superdense coding

In [ ]:
import numpy as np

I, X, Z = np.eye(2), np.array([[0, 1], [1, 0]]), np.diag([1, -1])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
CNOT = np.kron(np.diag([1, 0]), I) + np.kron(np.diag([0, 1]), X)
bell = np.array([1, 0, 0, 1]) / np.sqrt(2)

for message in ["00", "01", "10", "11"]:
    psi = bell.copy()
    if message[1] == "1":
        psi = np.kron(X, I) @ psi        # Alice acts on her qubit q0 only
    if message[0] == "1":
        psi = np.kron(Z, I) @ psi
    psi = np.kron(H, I) @ CNOT @ psi     # Bob decodes
    read = np.argmax(np.abs(psi) ** 2)
    print(f"sent {message} -> Bob reads {read:02b} with probability {np.abs(psi[read]) ** 2:.3f}")

Expected output:

```
sent 00 -> Bob reads 00 with probability 1.000
sent 01 -> Bob reads 01 with probability 1.000
sent 10 -> Bob reads 10 with probability 1.000
sent 11 -> Bob reads 11 with probability 1.000
```

### Solution 5.1.2: Why CNOT cannot clone a superposition

In [ ]:
import numpy as np

s = 1 / np.sqrt(2)
plus, zero = np.array([s, s]), np.array([1, 0])
CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]])

out = CNOT @ np.kron(plus, zero)
copy = np.kron(plus, plus)
print("CNOT |+>|0> =", out.round(4))
print("|+>|+>      =", copy.round(4))
print("overlap |<copy|out>|^2 =", round(abs(np.vdot(copy, out)) ** 2, 4))

Expected output:

```
CNOT |+>|0> = [0.7071 0.     0.     0.7071]
|+>|+>      = [0.5 0.5 0.5 0.5]
overlap |<copy|out>|^2 = 0.5
```

The output is the Bell state, not |+⟩|+⟩: their overlap is only 1/2. Linearity forces this, and the same argument rules out every possible copying machine.

### Solution 5.2.1: Bernstein–Vazirani in one query

In [ ]:
import numpy as np

n, s = 4, 0b1011
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Hn = H
for _ in range(n - 1):
    Hn = np.kron(Hn, H)

signs = np.array([(-1) ** (bin(s & x).count("1") % 2) for x in range(2 ** n)])
psi = Hn @ (signs * (Hn @ np.eye(2 ** n)[0]))    # Hadamards, one query, Hadamards
best = np.argmax(np.abs(psi) ** 2)
print(f"hidden s = {s:04b}, measured {best:04b} with probability {abs(psi[best]) ** 2:.3f}")

Expected output:

```
hidden s = 1011, measured 1011 with probability 1.000
```

### Solution 5.2.2: The Hadamard sign rule

In [ ]:
import numpy as np

n = 3
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
Hn = np.kron(np.kron(H, H), H)
rule = np.array([[(-1) ** (bin(x & y).count("1") % 2) for x in range(8)] for y in range(8)]) / np.sqrt(8)
print("H(x)H(x)H follows the sign rule:", np.allclose(Hn, rule))
x = 0b101
print("signs of H(x)H(x)H|101> on |000>, |001>, ..., |111>:", np.sign(Hn[:, x]).astype(int))

Expected output:

```
H(x)H(x)H follows the sign rule: True
signs of H(x)H(x)H|101> on |000>, |001>, ..., |111>: [ 1 -1  1 -1 -1  1 -1  1]
```

### Solution 5.3.1: How many rounds for 64 items?

In [ ]:
import numpy as np

N, marked = 64, 17
theta = np.arcsin(1 / np.sqrt(N))
print(f"theta = {theta:.4f} rad, best k is about {np.pi / (4 * theta) - 0.5:.2f}")
amps = np.full(N, 1 / np.sqrt(N))
for k in range(0, 13):
    if k > 0:
        amps[marked] *= -1
        amps = 2 * amps.mean() - amps
    print(f"k = {k:2d}: P(marked) = {amps[marked] ** 2:.3f}")

Expected output:

```
theta = 0.1253 rad, best k is about 5.77
k =  0: P(marked) = 0.016
k =  1: P(marked) = 0.135
k =  2: P(marked) = 0.344
k =  3: P(marked) = 0.591
k =  4: P(marked) = 0.816
k =  5: P(marked) = 0.964
k =  6: P(marked) = 0.997
k =  7: P(marked) = 0.907
k =  8: P(marked) = 0.718
k =  9: P(marked) = 0.475
k = 10: P(marked) = 0.238
k = 11: P(marked) = 0.066
k = 12: P(marked) = 0.000
```

The peak is at k = 6, with 99.7%. By k = 12 the probability has fallen to almost 0: the state has rotated past the answer to the far side.

### Solution 5.3.2: Several marked items

In [ ]:
import numpy as np

N = 64
marked = [3, 20, 41, 60]
M = len(marked)
amps = np.full(N, 1 / np.sqrt(N))
print(f"(pi/4) sqrt(N/M) = {np.pi / 4 * np.sqrt(N / M):.2f}")
for k in range(0, 7):
    if k > 0:
        amps[marked] *= -1
        amps = 2 * amps.mean() - amps
    print(f"k = {k}: P(any marked) = {np.sum(amps[marked] ** 2):.3f}")

Expected output:

```
(pi/4) sqrt(N/M) = 3.14
k = 0: P(any marked) = 0.062
k = 1: P(any marked) = 0.473
k = 2: P(any marked) = 0.908
k = 3: P(any marked) = 0.961
k = 4: P(any marked) = 0.582
k = 5: P(any marked) = 0.125
k = 6: P(any marked) = 0.020
```

### Solution 5.4.1: QFT|x⟩ is a product state

In [ ]:
import numpy as np

n, N = 3, 8
j, k = np.meshgrid(np.arange(N), np.arange(N))
QFT = np.exp(2j * np.pi * j * k / N) / np.sqrt(N)

all_match = True
for x in range(N):
    product = np.array([1.0])
    for q in range(n):
        qubit = np.array([1, np.exp(2j * np.pi * x / 2 ** (q + 1))]) / np.sqrt(2)
        product = np.kron(product, qubit)
    all_match = all_match and np.allclose(product, QFT[:, x])
print("QFT|x> is a product state for every x:", all_match)

Expected output:

```
QFT|x> is a product state for every x: True
```

Each qubit of the output sits on the equator of its Bloch sphere; as x counts up, q0 turns by half a turn per step, q1 by a quarter and q2 by an eighth.

### Solution 5.4.2: Phase estimation, outcome by outcome

In [ ]:
import numpy as np


def qft(M):
    j, k = np.meshgrid(np.arange(M), np.arange(M))
    return np.exp(2j * np.pi * j * k / M) / np.sqrt(M)


for phi, t in [(0.375, 3), (0.3, 3), (0.3, 5), (0.3, 7)]:
    M = 2 ** t
    register = np.exp(2j * np.pi * phi * np.arange(M)) / np.sqrt(M)   # after the kickbacks
    probs = np.abs(qft(M).conj().T @ register) ** 2                    # inverse QFT, then measure
    best = np.argmax(probs)
    print(f"phi = {phi}, t = {t}: most likely {best:0{t}b} = {best}/{M} = {best / M:.4f} with probability {probs[best]:.3f}")

Expected output:

```
phi = 0.375, t = 3: most likely 011 = 3/8 = 0.3750 with probability 1.000
phi = 0.3, t = 3: most likely 010 = 2/8 = 0.2500 with probability 0.578
phi = 0.3, t = 5: most likely 01010 = 10/32 = 0.3125 with probability 0.573
phi = 0.3, t = 7: most likely 0100110 = 38/128 = 0.2969 with probability 0.573
```

φ = 0.375 = 0.011 in binary is exact with three bits, so it comes out with certainty. For φ = 0.3 the nearest t-bit fraction wins with probability above 4/π² ≈ 0.405, and more counting qubits give a sharper estimate.

### Solution 5.4.3: The classical half of Shor's algorithm

In [ ]:
import math

for N, a in [(15, 7), (21, 2)]:
    r = 1
    while pow(a, r, N) != 1:
        r += 1
    half = pow(a, r // 2)
    print(f"N = {N}, a = {a}: powers {[pow(a, x, N) for x in range(1, r + 1)]}, period r = {r}")
    print(f"   gcd({half} - 1, {N}) = {math.gcd(half - 1, N)},  gcd({half} + 1, {N}) = {math.gcd(half + 1, N)}")

Expected output:

```
N = 15, a = 7: powers [7, 4, 13, 1], period r = 4
   gcd(49 - 1, 15) = 3,  gcd(49 + 1, 15) = 5
N = 21, a = 2: powers [2, 4, 8, 16, 11, 1], period r = 6
   gcd(8 - 1, 21) = 7,  gcd(8 + 1, 21) = 3
```

The quantum computer's only job in Shor's algorithm is finding r. Trying x one by one, as here, takes far too long when N has hundreds of digits.